# Pipeline DLT Gold - IBGE Geografia

**Modelagem Dimensional - Geografia Hierarquizada**

Dimensões de divisão territorial brasileira segundo o IBGE.

## Dimensões
* dim_regiao - Regiões do Brasil (N, NE, CO, SE, S)
* dim_estado - Estados e Distrito Federal (27 UFs)
* dim_municipio - Municípios brasileiros (5.570)
* dim_distrito - Distritos
* dim_subdistrito - Subdistritos
* dim_geografia_completa - View desnormalizada com hierarquia completa

## Dimensão: Regiões

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_regiao
(
  id_regiao BIGINT COMMENT 'Identificador único da região brasileira.',
  nome_regiao STRING COMMENT 'Nome da região (Norte, Nordeste, Centro-Oeste, Sudeste, Sul).',
  sigla_regiao STRING COMMENT 'Sigla da região (N, NE, CO, SE, S).',
  CONSTRAINT not_null_id_regiao EXPECT (id_regiao IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Dimensão de regiões brasileiras - nível mais alto da hierarquia geográfica.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT DISTINCT
  cr.id_regiao,
  cr.nome_regiao,
  cr.sigla_regiao
FROM silver_${var.environment}.ds_ibge.cleaned_regioes cr;

## Dimensão: Estados

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_estado
(
  id_estado BIGINT COMMENT 'Identificador único do estado (código IBGE).',
  nome_estado STRING COMMENT 'Nome completo do estado.',
  sigla_estado STRING COMMENT 'Sigla da UF (ex: SP, RJ, MG).',
  id_regiao BIGINT COMMENT 'Identificador da região à qual pertence (FK).',
  nome_regiao STRING COMMENT 'Nome da região.',
  sigla_regiao STRING COMMENT 'Sigla da região.',
  CONSTRAINT not_null_id_estado EXPECT (id_estado IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Dimensão de estados brasileiros (27 UFs), relacionada à dim_regiao.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT DISTINCT
  ce.id_estado,
  ce.nome_estado,
  ce.sigla_estado,
  ce.id_regiao,
  ce.nome_regiao,
  ce.sigla_regiao
FROM silver_${var.environment}.ds_ibge.cleaned_estados ce;

## Dimensão: Municípios

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_municipio
(
  id_municipio BIGINT COMMENT 'Identificador único do município (código IBGE de 7 dígitos).',
  nome_municipio STRING COMMENT 'Nome completo do município.',
  id_microrregiao BIGINT COMMENT 'Identificador da microrregião.',
  nome_microrregiao STRING COMMENT 'Nome da microrregião.',
  id_mesorregiao BIGINT COMMENT 'Identificador da mesorregião.',
  nome_mesorregiao STRING COMMENT 'Nome da mesorregião.',
  id_regiao_imediata BIGINT COMMENT 'Identificador da região imediata.',
  nome_regiao_imediata STRING COMMENT 'Nome da região imediata.',
  id_regiao_intermediaria BIGINT COMMENT 'Identificador da região intermediária.',
  nome_regiao_intermediaria STRING COMMENT 'Nome da região intermediária.',
  id_estado BIGINT COMMENT 'Identificador do estado (FK).',
  nome_estado STRING COMMENT 'Nome do estado.',
  sigla_estado STRING COMMENT 'Sigla da UF.',
  id_regiao BIGINT COMMENT 'Identificador da região (FK).',
  nome_regiao STRING COMMENT 'Nome da região.',
  sigla_regiao STRING COMMENT 'Sigla da região.',
  CONSTRAINT not_null_id_municipio EXPECT (id_municipio IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Dimensão de municípios brasileiros (5.570) com hierarquia geográfica completa.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT
  cm.id_municipio,
  cm.nome_municipio,
  cm.id_microrregiao,
  cm.nome_microrregiao,
  cm.id_mesorregiao,
  cm.nome_mesorregiao,
  cm.id_regiao_imediata,
  cm.nome_regiao_imediata,
  cm.id_regiao_intermediaria,
  cm.nome_regiao_intermediaria,
  cm.id_estado,
  cm.nome_estado,
  cm.sigla_estado,
  cm.id_regiao,
  cm.nome_regiao,
  cm.sigla_regiao
FROM silver_${var.environment}.ds_ibge.cleaned_municipios cm;

## Dimensão: Distritos

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_distrito
(
  id_distrito BIGINT COMMENT 'Identificador único do distrito.',
  nome_distrito STRING COMMENT 'Nome do distrito.',
  id_municipio BIGINT COMMENT 'Identificador do município ao qual pertence (FK).',
  nome_municipio STRING COMMENT 'Nome do município.',
  id_estado BIGINT COMMENT 'Identificador do estado.',
  nome_estado STRING COMMENT 'Nome do estado.',
  sigla_estado STRING COMMENT 'Sigla da UF.',
  id_regiao BIGINT COMMENT 'Identificador da região.',
  nome_regiao STRING COMMENT 'Nome da região.',
  CONSTRAINT not_null_id_distrito EXPECT (id_distrito IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Dimensão de distritos brasileiros, subdivisões dos municípios.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT
  cd.id_distrito,
  cd.nome_distrito,
  cd.id_municipio,
  cd.nome_municipio,
  cd.id_estado,
  cd.nome_estado,
  cd.sigla_estado,
  cd.id_regiao,
  cd.nome_regiao
FROM silver_${var.environment}.ds_ibge.cleaned_distritos cd;

## Dimensão: Subdistritos

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_subdistrito
(
  id_subdistrito BIGINT COMMENT 'Identificador único do subdistrito.',
  nome_subdistrito STRING COMMENT 'Nome do subdistrito.',
  id_distrito BIGINT COMMENT 'Identificador do distrito ao qual pertence (FK).',
  nome_distrito STRING COMMENT 'Nome do distrito.',
  id_municipio BIGINT COMMENT 'Identificador do município.',
  nome_municipio STRING COMMENT 'Nome do município.',
  id_estado BIGINT COMMENT 'Identificador do estado.',
  sigla_estado STRING COMMENT 'Sigla da UF.',
  CONSTRAINT not_null_id_subdistrito EXPECT (id_subdistrito IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT 'Dimensão de subdistritos brasileiros, menor unidade da hierarquia geográfica.'
TBLPROPERTIES (
  "quality" = "gold",
  pipelines.autoOptimize.managed = true,
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact = true
)
CLUSTER BY AUTO
AS
SELECT
  cs.id_subdistrito,
  cs.nome_subdistrito,
  cs.id_distrito,
  cs.nome_distrito,
  cs.id_municipio,
  cs.nome_municipio,
  cs.id_estado,
  cs.sigla_estado
FROM silver_${var.environment}.ds_ibge.cleaned_subdistritos cs;